# 02 — Single star, 10 000-row trail

Three configurations for a single bright star with no sky background or read noise:

| # | PSF | Jitter |
|---|-----|--------|
| 1 | No  | No     |
| 2 | Yes | No     |
| 3 | Yes | Yes (OU) |

Diagnostics shown for each: full trail image, column cross-sections at several rows,
and (for the jittered case) the centroid track.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from startrail.simulate.trail import make_trail

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
SHAPE  = (10_000, 100)   # (rows, cols) — 10 000-row CCD readout
CX     = 50.0            # star x position (column centre)
CY     = 5_000.0         # star y position (row centre)
LENGTH = 9_800.0         # trail length in pixels (fills almost the full detector)
FWHM   = 3.0             # PSF FWHM in pixels
FLUX   = 1e8             # total trail flux (counts)
TAU    = 500.0           # OU guider timescale (pixels along trail)
JITTER = 1.5             # steady-state jitter RMS (pixels)
SEED   = 42

# Config 1: no PSF (sub-pixel FWHM ≈ delta function), no jitter
trail_1 = make_trail(SHAPE, CX, CY, angle=0.0, length=LENGTH,
                     fwhm=0.5, flux=FLUX)

# Config 2: PSF, no jitter
trail_2 = make_trail(SHAPE, CX, CY, angle=0.0, length=LENGTH,
                     fwhm=FWHM, flux=FLUX)

# Config 3: PSF, OU jitter
trail_3 = make_trail(SHAPE, CX, CY, angle=0.0, length=LENGTH,
                     fwhm=FWHM, flux=FLUX,
                     jitter_sigma=JITTER, tau=TAU, seed=SEED)

trails = [trail_1, trail_2, trail_3]
labels = [
    'Config 1\nNo PSF, no jitter',
    'Config 2\nPSF, no jitter',
    f'Config 3\nPSF + OU jitter\n(σ={JITTER} px, τ={TAU:.0f} px)',
]
print('Trails generated.')
for t, lbl in zip(trails, labels):
    print(f"  {lbl.splitlines()[0]:25s}  flux={t.sum():.3e}  peak={t.max():.3e}")

## 1. Full-trail images

Each image is 10 000 × 100 px, displayed with `aspect='auto'` so the
full readout direction is visible.  The colour scale is linear and
clipped at the 99th percentile of non-zero pixels.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 9))

for ax, trail, label in zip(axes, trails, labels):
    nonzero = trail[trail > 0]
    vmax = np.percentile(nonzero, 99) if nonzero.size else 1.0
    ax.imshow(trail, origin='lower', cmap='hot',
              vmin=0, vmax=vmax, aspect='auto')
    ax.set_title(label, fontsize=9)
    ax.set_xlabel('column (px)')
    ax.set_ylabel('row (px)')

plt.suptitle('10 000-row single-star trail — three configurations', fontsize=12)
plt.tight_layout()
plt.savefig('../plots/02_full_trails.png', dpi=120)
plt.show()

## 2. Column cross-sections

Pixel value vs column at three representative rows:
near the bottom (row 600), middle (row 5 000), and near the top (row 9 400).
Config 1 should show a near-delta-function; Config 2 a stable Gaussian;
Config 3 a Gaussian whose centre shifts with the jitter.

In [ ]:
sample_rows = [600, 5_000, 9_400]
cols = np.arange(SHAPE[1])

fig, axes = plt.subplots(len(trails), len(sample_rows),
                         figsize=(11, 7), sharey='row')

for row_axes, trail, label in zip(axes, trails, labels):
    peak = trail.max()
    for ax, r in zip(row_axes, sample_rows):
        ax.plot(cols, trail[r] / peak, lw=1.2)
        ax.axvline(CX, color='k', lw=0.5, ls='--', alpha=0.4)
        ax.set_title(f'row {r:,}', fontsize=9)
        ax.set_xlabel('column (px)')
        ax.set_xlim(CX - 20, CX + 20)
    row_axes[0].set_ylabel(f'{label.splitlines()[0]}\nnorm. counts', fontsize=8)

plt.suptitle('Column cross-sections at three row positions', fontsize=12)
plt.tight_layout()
plt.savefig('../plots/02_cross_sections.png', dpi=120)
plt.show()

## 3. Row-summed flux profile

Total counts per row along the trail.  Should be flat (uniform flux
per readout step) for Configs 1 and 2.  Config 3 may show edge
effects where jitter pushes flux slightly outside the window.

In [ ]:
rows = np.arange(SHAPE[0])

fig, ax = plt.subplots(figsize=(11, 3))
for trail, label in zip(trails, labels):
    row_flux = trail.sum(axis=1)
    ax.plot(rows, row_flux / row_flux.max(), lw=0.8,
            label=label.replace('\n', ' — '))

ax.set_xlabel('row (px)')
ax.set_ylabel('norm. counts per row')
ax.set_title('Row-summed flux profile')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('../plots/02_row_flux.png', dpi=120)
plt.show()

## 4. Centroid track (Config 3 — jitter)

Flux-weighted x centroid at each row, revealing the OU jitter path.
The dashed lines mark ±σ_jitter around the nominal centre.

In [ ]:
row_sum = trail_3.sum(axis=1)
# Only compute centroid where the trail has meaningful signal
trail_rows = np.where(row_sum > row_sum.max() * 1e-4)[0]

centroids = np.full(SHAPE[0], np.nan)
centroids[trail_rows] = (
    (trail_3[trail_rows] * cols[np.newaxis, :]).sum(axis=1)
    / row_sum[trail_rows]
)

fig, ax = plt.subplots(figsize=(11, 3))
ax.plot(trail_rows, centroids[trail_rows] - CX, lw=0.7, label='centroid offset')
ax.axhline( JITTER, color='r', lw=0.8, ls='--', label=f'±{JITTER} px (target σ)')
ax.axhline(-JITTER, color='r', lw=0.8, ls='--')
ax.axhline(0, color='k', lw=0.5, ls=':')
ax.set_xlabel('row (px)')
ax.set_ylabel('centroid offset (px)')
ax.set_title(f'Config 3 — jitter centroid track  (σ={JITTER} px, τ={TAU:.0f} px)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('../plots/02_centroid_track.png', dpi=120)
plt.show()

rms = np.nanstd(centroids[trail_rows] - CX)
print(f'Centroid RMS offset: {rms:.3f} px  (target σ = {JITTER} px)')
print(f'Centroid range:      [{np.nanmin(centroids[trail_rows]-CX):.3f},'
      f'  {np.nanmax(centroids[trail_rows]-CX):.3f}] px')